In [30]:
from qiskit.quantum_info import Pauli, Clifford
from qiskit.circuit.library import *
from qiskit import QuantumCircuit

p = Pauli("IXI")       # 标签（注意 qubit 顺序）
q = Pauli("-iYYI")

pq = p @ q             # 张量积
# prod = p * q           # Pauli 乘（不追踪相位）/ 用 sgn_prod 追踪相位
comm = p.commutes(q)   # 是否对易


In [31]:
qc = QuantumCircuit(3)
qc.append(iSwapGate(), [0, 1])

p.evolve(qc) # default: frame='h'

Pauli('-IZY')

In [32]:
p.evolve(Clifford(qc), frame='s')

Pauli('IZY')

In [ ]:
instr = Clifford(iSwapGate()).to_circuit()

In [13]:
instr

Instruction(name='Clifford: Stabilizer = ['+ZI', '+IZ'], Destabilizer = ['+YZ', '+ZY']', num_qubits=3, num_clbits=0, params=[])

In [2]:
from qiskit.quantum_info import PauliList

plist = PauliList(["XZI", "IYY", "ZZI"])
plist[0]          # -> Pauli
plist[1:]         # -> PauliList
plist.to_labels() # ["XZI", "IYY", "ZZI"]


['XZI', 'IYY', 'ZZI']

In [3]:
p0 = Pauli("XZI")
p1 = Pauli("-iZZI")
p2 = Pauli("-XZY")
p3 = Pauli("iYIX")
print(p0.phase, p1.phase, p2.phase, p3.phase)  # 0, 1, 2, 3

0 1 2 3


In [4]:
from qiskit.quantum_info import SparsePauliOp

H = SparsePauliOp.from_list([
    ("ZZI", 1.2),
    ("IXX", -0.7),
    ("IXX", 0.3),
])

H2 = 0.5 * H + SparsePauliOp.from_list([("XZI", 2.0)])
H.simplify()   # 合并重复 Pauli
print(H.paulis)      # -> PauliList
print(H.coeffs)       # -> ndarray

['ZZI', 'IXX', 'IXX']
[ 1.2+0.j -0.7+0.j  0.3+0.j]


In [8]:
H.paulis.z

array([[False,  True,  True],
       [False, False, False],
       [False, False, False]])

In [9]:
from phoenix_qiskit import Hamiltonian
from phoenix_qiskit.primitives import grouping

/Users/anan/git-projects/quantum/phoenix/phoenix_qiskit/basics.py:44: SyntaxWarning: invalid escape sequence '\d'
  """


In [10]:
h = Hamiltonian(H.paulis, H.coeffs)


In [11]:
h.group_paulis()

{(0,
  1): SparsePauliOp(['ZZI'],
               coeffs=[1.2+0.j]),
 (1,
  2): SparsePauliOp(['IXX'],
               coeffs=[-0.4+0.j])}

In [9]:
grouping.group_paulis_and_coeffs(['IXX', 'IYY', 'ZIY'], [0.1,0.2,0.3])

{(0, 2): (['ZIY'], array([0.3])), (1, 2): (['IXX', 'IYY'], array([0.1, 0.2]))}

In [10]:
h.print_tableau()

+-------+--------+--------+------+
| Pauli | X part | Z part | sign |
+-------+--------+--------+------+
|  ZZI  | 0 0 0  | 0 1 1  |  0   |
|  IXX  | 1 1 0  | 0 0 0  |  0   |
+-------+--------+--------+------+


In [16]:
h.paulis

PauliList(['ZZI', 'IXX'])

In [11]:
import numpy as np

weights = np.sum(np.logical_or(h.paulis.x, h.paulis.z), axis=1)
local_mask = weights <= 1

In [13]:
local_mask

array([False, False])

array([False, False])

In [ ]:
np.hstack([h.paulis.x, h.paulis.z, np.expand_dims(h.paulis.phase, axis=-1)])

array([[0, 0, 0, 0, 1, 1, 0],
       [1, 1, 0, 0, 0, 0, 0]])

In [18]:
h.with_ops.sum(axis=1)

array([2, 2])

In [22]:
h.with_ops.sum(axis=0) > 0

array([ True,  True,  True])

In [26]:
from itertools import combinations

qubit_pairs = sorted(combinations([0,1,3,4], 2), key=lambda idx: (idx[0] % 2))
qubit_pairs

[(0, 1), (0, 3), (0, 4), (1, 3), (1, 4), (3, 4)]

In [34]:
from qiskit.circuit.library import HGate, IGate
from qiskit.quantum_info import Clifford

ham = Hamiltonian(['YY', 'ZY'], [1.1, 2.2])
p = ham.paulis.evolve(Clifford(HGate()).tensor(Clifford(IGate())))
p

PauliList(['-YY', 'XY'])

In [37]:
p.phase

array([2, 0])

In [59]:
from qiskit.quantum_info import Clifford
from qiskit.circuit.library import HGate, SGate

H = Clifford(HGate())
S = Clifford(SGate())

# H.compose(S) 表示：先 H，再 S
# 等价于电路 q --[H]--[S]--
SH = H.compose(S)

# 对比：S.compose(H) 表示：先 S，再 H
# 等价于电路 q --[S]--[H]--
HS = S.compose(H)

assert np.allclose(HS.to_matrix(), HGate().to_matrix() @ SGate().to_matrix())
assert np.allclose(SH.to_matrix(), SGate().to_matrix() @ HGate().to_matrix())

In [56]:
HGate().to_matrix() @ SGate().to_matrix()

array([[0.70710678+0.j        , 0.        +0.70710678j],
       [0.70710678+0.j        , 0.        -0.70710678j]])

In [57]:
HS.to_matrix() == HGate().to_matrix() @ SGate().to_matrix()

array([[ True,  True],
       [ True,  True]])

In [46]:
HS.to_matrix()

array([[0.70710678+0.j        , 0.        +0.70710678j],
       [0.70710678+0.j        , 0.        -0.70710678j]])

In [48]:
HGate().to_matrix() @ SGate().to_matrix()

array([[0.70710678+0.j        , 0.        +0.70710678j],
       [0.70710678+0.j        , 0.        -0.70710678j]])

Clifford(array([[ True,  True,  True],
       [ True, False, False]]))

In [24]:
import numpy as np
np.any(h.with_ops, axis=0)

array([ True,  True,  True])

In [13]:
h.group_paulis()

{(0,
  1): SparsePauliOp(['ZZI'],
               coeffs=[1.2+0.j]),
 (1,
  2): SparsePauliOp(['IXX'],
               coeffs=[-0.4+0.j])}

In [14]:
h.group_same_weights()

[SparsePauliOp(['ZZI'],
               coeffs=[1.2+0.j]),
 SparsePauliOp(['IXX'],
               coeffs=[-0.4+0.j])]

In [49]:
from phoenix_qiskit import CNOTEquivCliffordGate
import qiskit.quantum_info as qi

p0 = 'I'
p1 = 'X'
op = qi.SparsePauliOp.from_list([
    ("II", 0.5),
    (f"I{p0}", 0.5),
    (f"{p1}I", 0.5),
    (f"{p1}{p0}", -0.5)
])


In [51]:
op.to_matrix()

array([[1.+0.j, 0.+0.j, 0.+0.j, 0.+0.j],
       [0.+0.j, 1.+0.j, 0.+0.j, 0.+0.j],
       [0.+0.j, 0.+0.j, 1.+0.j, 0.+0.j],
       [0.+0.j, 0.+0.j, 0.+0.j, 1.+0.j]])

In [32]:
SparsePauliOp(['XI', 'IY'], [0.5+0.j, 0. +0.5j])

SparsePauliOp(['XI', 'IY'],
              coeffs=[0.5+0.j , 0. +0.5j])

In [38]:
SparsePauliOp(['XI', 'IY'], [0.5+0.j, 0. +0.5j]).paulis.x

array([[False,  True],
       [ True, False]])

In [ ]:
SparsePauliOp.from_list([('XI', 0.5+0.j), ('IY', 0. +0.5j)]) # (['XI', 'IY'], [0.5+0.j, 0. +0.5j])

SparsePauliOp(['XI', 'IY'],
              coeffs=[0.5+0.j , 0. +0.5j])

In [ ]:
from qiskit.quantum_info import Clifford, Pauli
from qiskit.circuit.library import iSwapGate, CXGate, SGate, TGate

U = Clifford.from_operator(iSwapGate())   # 从 Clifford 电路得到 tableau
p = Pauli("XZI")

# p2 = U.conjugate(p)             # U p U^\dagger
# U_inv = U.adjoint()
# qc2 = U.to_circuit()


In [ ]:
Clifford(TGate())

# Clifford(array([[ True,  True, False, False, False],
#        [False,  True, False, False, False],
#        [False, False,  True, False, False],
#        [False, False,  True,  True, False]]))

QiskitError: 'Cannot update Clifford with non-Clifford gate t'

In [ ]:
Clifford(CXGate()).to_matrix()

array([[1.+0.j, 0.+0.j, 0.+0.j, 0.+0.j],
       [0.+0.j, 0.+0.j, 0.+0.j, 1.+0.j],
       [0.+0.j, 0.+0.j, 1.+0.j, 0.+0.j],
       [0.+0.j, 1.+0.j, 0.+0.j, 0.+0.j]])

array([[1.+0.j, 0.+0.j, 0.+0.j, 0.+0.j],
       [0.+0.j, 0.+0.j, 0.+0.j, 1.+0.j],
       [0.+0.j, 0.+0.j, 1.+0.j, 0.+0.j],
       [0.+0.j, 1.+0.j, 0.+0.j, 0.+0.j]])

In [ ]:
Clifford.from_operator(iSwapGate())

Clifford(array([[False,  True,  True,  True, False],
       [ True, False,  True,  True, False],
       [False, False, False,  True, False],
       [False, False,  True, False, False]]))

In [ ]:
from qiskit.quantum_info import Pauli, Clifford
from qiskit.circuit.library import CXGate

U = Clifford(CXGate())  # control=0,target=1

print(U.conjugate(Pauli("XI")))  # X on qubit-0
# 预期标签显示为 "XX"（右边是 q0），即 X0 -> X0 X1

print(U.conjugate(Pauli("IZ")))  # Z on qubit-0
# 预期为 "ZZ"?（具体看门方向），和 tableau 的 x/z 列一致


TypeError: Clifford.conjugate() takes 1 positional argument but 2 were given

In [29]:
Pauli('XI').evolve(CXGate())

Pauli('XI')

In [31]:
Clifford(CXGate()).to_instruction()

Instruction(name='Clifford: Stabilizer = ['+IZ', '+ZZ'], Destabilizer = ['+XX', '+XI']', num_qubits=2, num_clbits=0, params=[])

In [ ]:
from qiskit.quantum_info import Pauli, PauliList, Clifford
from qiskit.circuit.library import CXGate

# 1. 创建 Clifford 算符 (CNOT)
cliff = Clifford(CXGate())

# 2. 单个 Pauli 演化 (evolve)
p = Pauli("XI")  # X on q1
p_evolved = p.evolve(cliff)
print(f"Pauli evolve: {p} -> {p_evolved}") 

# 3. PauliList 演化 (批量处理)
plist = PauliList(["XI", "IZ"])
plist_evolved = plist.evolve(cliff)
print(f"PauliList evolve: {plist} -> {plist_evolved}")


Pauli evolve: XI -> XI
PauliList evolve: ['XI', 'IZ'] -> ['XI', 'IZ']


In [ ]:
plist.x

array([[False, False,  True],
       [ True,  True, False],
       [False, False, False]])

In [ ]:
plist.to_labels()

['XZI', 'IYY', 'ZZI']

In [ ]:
# XZI, IYY, ZZI
PauliList.from_symplectic(plist.x, plist.z)

PauliList(['ZXI', 'IYY', 'XXI'])

In [ ]:
print(plist.to_labels())
plist.x

['XZI', 'IYY', 'ZZI']


array([[False, False,  True],
       [ True,  True, False],
       [False, False, False]])

In [48]:
H = SparsePauliOp.from_list([("-XY", np.pi), ("ZZ", 0.5)])


In [50]:
h

SparsePauliOp(['ZZI', 'IXX'],
              coeffs=[ 0. +1.2j, -0.4+0.j ])

In [ ]:
from qiskit.circuit import QuantumCircuit
from qiskit.circuit.library import PauliEvolutionGate
from qiskit.synthesis import LieTrotter, SuzukiTrotter, ProductFormula, EvolutionSynthesis

# 1. Define Hamiltonian as SparsePauliOp
H = SparsePauliOp.from_list([("-XY", np.pi), ("ZZ", 0.5)])

# 2. Create  (exp(-iHt))
# This gate represents the unitary evolution U(t) = e^{-iHt}
evo_gate = PauliEvolutionGate(H, time=1.0)

# 3. Create QuantumCircuit and append the gate
qc = QuantumCircuit(2)
qc.append(evo_gate, [0, 1])

# 4. Synthesize (decompose) into basis gates
# By default, Qiskit might use LieTrotter or MatrixExponential depending on context/settings
# We can explicitly decompose it using a synthesis plugin or .decompose()
print("Original Circuit (High-level):")
print(qc.draw())

print("\nDecomposed Circuit (LieTrotter):")
# Using LieTrotter synthesis
trotter_factory = LieTrotter()
qc_synthesized = trotter_factory.synthesize(evo_gate)
qc_synthesized.draw(fold=-1)

Original Circuit (High-level):
     ┌────────────────────────┐
q_0: ┤0                       ├
     │  exp(-it (XY + ZZ))(1) │
q_1: ┤1                       ├
     └────────────────────────┘

Decomposed Circuit (LieTrotter):


┌────┐┌───┐┌─────────┐┌───┐┌──────┐        
q_0: ┤ √X ├┤ X ├┤ Rz(-2π) ├┤ X ├┤ √Xdg ├─■──────
     ├───┬┘└─┬─┘└─────────┘└─┬─┘└┬───┬─┘ │ZZ(1) 
q_1: ┤ H ├───■───────────────■───┤ H ├───■──────
     └───┘                       └───┘

In [40]:
h

SparsePauliOp(['ZZI', 'IXX'],
              coeffs=[ 0. +1.2j, -0.4+0.j ])

In [43]:
h.tableau()

array([[0, 0, 0, 0, 1, 1],
       [1, 1, 0, 0, 0, 0]])

In [42]:
h.print_tableau()

+-------+--------+--------+------+
| Pauli | X part | Z part | sign |
+-------+--------+--------+------+
|  ZZI  | 0 0 0  | 0 1 1  |  0   |
|  IXX  | 1 1 0  | 0 0 0  |  0   |
+-------+--------+--------+------+


In [44]:
h.paulis[0][0]

Pauli('I')

In [46]:
h.paulis.z

array([[False,  True,  True],
       [False, False, False]])

In [66]:
qc = QuantumCircuit(4)
qc.append(PauliEvolutionGate(h), [1,2,3])
qc.draw(fold=-1)

q_0: ────────────────────────────
     ┌──────────────────────────┐
q_1: ┤0                         ├
     │                          │
q_2: ┤1 exp(-it (ZZI + IXX))(1) ├
     │                          │
q_3: ┤2                         ├
     └──────────────────────────┘

In [65]:
h.paulis.to_labels()

['ZZI', 'IXX']

In [68]:
h.group_commuting()

[SparsePauliOp(['ZZI'],
               coeffs=[1.2+0.j]),
 SparsePauliOp(['IXX'],
               coeffs=[-0.4+0.j])]

In [70]:
h.group_paulis()

{(0,
  1): SparsePauliOp(['ZZI'],
               coeffs=[1.2+0.j]),
 (1,
  2): SparsePauliOp(['IXX'],
               coeffs=[-0.4+0.j])}

In [ ]:
qc.decompose().draw()

┌────┐┌───┐┌────────┐┌───┐┌──────┐        
q_0: ┤ √X ├┤ X ├┤ Rz(-2) ├┤ X ├┤ √Xdg ├─■──────
     ├───┬┘└─┬─┘└────────┘└─┬─┘└┬───┬─┘ │ZZ(1) 
q_1: ┤ H ├───■──────────────■───┤ H ├───■──────
     └───┘                      └───┘

In [ ]:
from phoenix_qiskit import basics

In [ ]:
basics.CNOTEquivCliffordGate('Z', 'Z')._build_circuit().draw()

q_0: ───────■───────
     ┌───┐┌─┴─┐┌───┐
q_1: ┤ H ├┤ X ├┤ H ├
     └───┘└───┘└───┘

In [61]:
from qiskit.quantum_info import Clifford
from qiskit.circuit.library import CXGate

cliff = Clifford(CXGate())

In [63]:
cliff.to_labels()

['+XX', '+XI', '+IZ', '+ZZ']

In [73]:
cliff.to_labels(mode='D')

['+XX', '+XI']

In [77]:
cliff.to_labels(mode='S')

['+IZ', '+ZZ']

In [79]:
cliff.to_dict()

{'stabilizer': ['+IZ', '+ZZ'], 'destabilizer': ['+XX', '+XI']}